# YOLO Mask Overlay Notebook

Lightweight pipeline:
1. Upload a YOLO model (`.pt`) - prints the available classes.
2. Upload an MP4.
3. Pick which classes to keep.
4. Choose a temporal smoothing window.
5. Render an output MP4 with segmentation masks overlaid only for the selected classes. Masks are temporally smoothed across the chosen window.

In [ ]:
# Install deps (skip if already present)
%pip install -q ultralytics opencv-python ipywidgets numpy

In [ ]:
import os
import tempfile
from collections import defaultdict, deque
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Video, FileLink, clear_output
from ultralytics import YOLO

WORK_DIR = Path(tempfile.mkdtemp(prefix="yolo_overlay_"))
STATE = {"model": None, "model_path": None, "video_path": None, "names": {}}
print(f"Working directory: {WORK_DIR}")

## 1. Upload YOLO model and inspect classes

In [ ]:
model_uploader = widgets.FileUpload(accept=".pt", multiple=False, description="Upload .pt")
model_status = widgets.Output()

def _on_model_upload(change):
    with model_status:
        clear_output()
        if not model_uploader.value:
            return
        item = list(model_uploader.value)[0]
        if isinstance(item, dict):
            name, content = item["name"], item["content"]
        else:
            name, content = item, model_uploader.value[item]["content"]
        path = WORK_DIR / name
        path.write_bytes(content)
        STATE["model_path"] = str(path)
        STATE["model"] = YOLO(str(path))
        STATE["names"] = STATE["model"].names
        print(f"Loaded {name}")
        print("\nAvailable classes:")
        for idx, cname in STATE["names"].items():
            print(f"  {idx:>3}: {cname}")
        task = getattr(STATE["model"], "task", None)
        if task and task != "segment":
            print(f"\nWarning: model task is '{task}'. Pixel masks require a segmentation model (e.g. yolov8n-seg.pt).")
            print("Detection-only models will fall back to filled bounding boxes.")

model_uploader.observe(_on_model_upload, names="value")
display(model_uploader, model_status)

## 2. Upload video

In [ ]:
video_uploader = widgets.FileUpload(accept=".mp4,.mov,.avi,.mkv", multiple=False, description="Upload video")
video_status = widgets.Output()

def _on_video_upload(change):
    with video_status:
        clear_output()
        if not video_uploader.value:
            return
        item = list(video_uploader.value)[0]
        if isinstance(item, dict):
            name, content = item["name"], item["content"]
        else:
            name, content = item, video_uploader.value[item]["content"]
        path = WORK_DIR / name
        path.write_bytes(content)
        STATE["video_path"] = str(path)
        cap = cv2.VideoCapture(str(path))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        print(f"Loaded {name}: {w}x{h}, {fps:.2f} fps, {n} frames")

video_uploader.observe(_on_video_upload, names="value")
display(video_uploader, video_status)

## 3. Pick classes\n\nType the class names or ids you want to keep, comma-separated (e.g. `person, car` or `0, 2`). Case-insensitive.

In [ ]:
if not STATE["names"]:\n    raise RuntimeError("Upload a model first.")\n\n# Type class names or ids, comma-separated\nKEEP = "person, car"\n\n_name_to_id = {n.lower(): i for i, n in STATE["names"].items()}\nselected_ids = []\nfor token in KEEP.split(","):\n    t = token.strip().lower()\n    if not t:\n        continue\n    if t.isdigit() and int(t) in STATE["names"]:\n        selected_ids.append(int(t))\n    elif t in _name_to_id:\n        selected_ids.append(_name_to_id[t])\n    else:\n        raise ValueError(f"Unknown class: {token!r}")\nselected_ids = sorted(set(selected_ids))\nprint("Keeping:", [(i, STATE["names"][i]) for i in selected_ids])

## 4. Run inference and write output

In [ ]:
def _color_for(class_id):\n    rng = np.random.default_rng(int(class_id) * 9973 + 7)\n    return tuple(int(c) for c in rng.integers(64, 255, size=3))\n\nSMOOTH_WINDOW = 5\nCONF = 0.25\nALPHA = 0.5\n\ndef run(model, video_path, selected_ids, window, conf, alpha, out_path):\n    selected_ids = set(int(i) for i in selected_ids)\n    if not selected_ids:\n        raise ValueError("Select at least one class.")\n\n    cap = cv2.VideoCapture(video_path)\n    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    cap.release()\n\n    fourcc = cv2.VideoWriter_fourcc(*"mp4v")\n    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))\n\n    history = defaultdict(lambda: deque(maxlen=max(1, int(window))))\n    is_seg = getattr(model, "task", None) == "segment"\n    names = STATE["names"]\n\n    progress = widgets.IntProgress(value=0, min=0, max=max(total, 1), description="Rendering:")\n    label = widgets.Label(value="")\n    display(widgets.HBox([progress, label]))\n\n    frame_idx = 0\n    for result in model.track(source=video_path, stream=True, persist=True, conf=conf, classes=list(selected_ids), verbose=False):\n        frame = result.orig_img.copy()\n        boxes = result.boxes\n        masks = result.masks if is_seg else None\n        overlay = np.zeros_like(frame, dtype=np.float32)\n        weight = np.zeros((h, w), dtype=np.float32)\n        labels_to_draw = []\n\n        if boxes is not None and len(boxes) > 0:\n            cls = boxes.cls.cpu().numpy().astype(int)\n            ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else np.arange(len(cls))\n            xyxy = boxes.xyxy.cpu().numpy().astype(int)\n            mask_data = masks.data.cpu().numpy() if masks is not None else None\n\n            for i in range(len(cls)):\n                c = int(cls[i])\n                if c not in selected_ids:\n                    continue\n                track_id = int(ids[i])\n                key = (track_id, c)\n\n                if mask_data is not None:\n                    m = mask_data[i]\n                    if m.shape != (h, w):\n                        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)\n                    m = m.astype(np.float32)\n                else:\n                    x1, y1, x2, y2 = xyxy[i]\n                    m = np.zeros((h, w), dtype=np.float32)\n                    m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 1.0\n\n                history[key].append(m)\n                smoothed = np.mean(np.stack(history[key], axis=0), axis=0)\n                binary = (smoothed >= 0.5).astype(np.float32)\n\n                color = np.array(_color_for(c), dtype=np.float32)\n                overlay += binary[..., None] * color\n                weight += binary\n\n                ys, xs = np.where(binary > 0)\n                if ys.size > 0:\n                    cx, cy = int(xs.mean()), int(ys.mean())\n                    labels_to_draw.append((cx, cy, names.get(c, str(c)), tuple(int(v) for v in color)))\n\n        if weight.max() > 0:\n            w_safe = np.clip(weight, 1.0, None)[..., None]\n            color_layer = overlay / w_safe\n            mask_any = (weight > 0)[..., None].astype(np.float32)\n            blended = frame.astype(np.float32) * (1 - alpha * mask_any) + color_layer * (alpha * mask_any)\n            frame = np.clip(blended, 0, 255).astype(np.uint8)\n\n        for cx, cy, text, color in labels_to_draw:\n            font = cv2.FONT_HERSHEY_SIMPLEX\n            scale = 0.6\n            thickness = 2\n            (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)\n            x = max(0, min(w - tw - 4, cx - tw // 2))\n            y = max(th + 4, min(h - 4, cy + th // 2))\n            cv2.rectangle(frame, (x - 2, y - th - 4), (x + tw + 2, y + baseline), (0, 0, 0), -1)\n            cv2.putText(frame, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)\n\n        writer.write(frame)\n        frame_idx += 1\n        if frame_idx % 5 == 0 or frame_idx == total:\n            progress.value = min(frame_idx, progress.max)\n            label.value = f"frame {frame_idx}/{total}"\n\n    writer.release()\n    progress.value = progress.max\n    label.value = f"done ({frame_idx} frames)"\n    return out_path\n\nif STATE["model"] is None or STATE["video_path"] is None:\n    raise RuntimeError("Upload both model and video first.")\n\nout_path = WORK_DIR / "output_overlay.mp4"\nrun(\n    model=STATE["model"],\n    video_path=STATE["video_path"],\n    selected_ids=selected_ids,\n    window=SMOOTH_WINDOW,\n    conf=CONF,\n    alpha=ALPHA,\n    out_path=out_path,\n)\nprint(f"Wrote {out_path}")

## 5. Preview / download

In [ ]:
out_path = WORK_DIR / "output_overlay.mp4"
display(FileLink(str(out_path)))
Video(str(out_path), embed=False)